# Stage-z drift QC

Each movie has a HAL `.off` focus-lock file next to it: one row per frame,
columns `frame offset power stage-z good-offset`. The lock should hold
`stage-z` constant over a FOV's stack.

For every round and FOV this notebook:
- reads `stage-z` from the `.off` file (files still being written are skipped
  until a later run),
- checks that all rows are equal, else records the first, min and max,
- caches the result, so a re-run reads only new `.off` files,
- plots the first-frame `stage-z` against `fov_id`, one line per round on the
  same axis, so drift between rounds shows as vertical offset.

Re-run any time to refresh.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.plots.experiment_plots import get_merci_figures_dir
from MERci.analysis.stage_z import (
    update_stage_z_cache, update_focustest_stage_z_cache, assign_x_positions, round_label,
    positions_to_grid_indices,
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # same as HAL writes
NOTEBOOK_NAME = "stage_z_drift"   # cache and figures folder name

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# Cache under analysis/cache/<notebook_name>/.
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / "stage_z_summary.csv"

# Figures under figures/MERci/during_imaging/{NOTEBOOK_NAME}/.
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"Positions tag: {POSITIONS_TAG}")
print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Cache  : {CACHE_PATH}")
print(f"Figures: {FIGURES_DIR}")

## 3 — Read / update the stage-z cache

Re-run this cell any time — it only reads `.off` files not already in the cache.

In [ ]:
n_before = 0
if CACHE_PATH.exists():
    import pandas as pd
    n_before = len(pd.read_csv(CACHE_PATH))

cache = update_stage_z_cache(config, meta, CACHE_PATH)

# Also the focus-lock test's .off files (data/focus_test/, not in round_info.csv).
# Empty if there was no focus test.
cache = update_focustest_stage_z_cache(config.data_dir / "focus_test", CACHE_PATH)

print(f"Stage-z cache: {len(cache)} FOV\u00d7series entries total "
      f"({len(cache) - n_before} newly read this run).")

drifted = cache[~cache["all_same"]]
if drifted.empty:
    print("Every FOV's stage-z was constant across its whole frame stack.")
else:
    print(f"{len(drifted)} FOV(s) had a non-constant stage-z within their stack:")
    display(
        drifted[["round_id", "fov_id", "series", "first_stage_z", "min_stage_z", "max_stage_z"]]
        .sort_values(["round_id", "fov_id"])
    )

## 4 — Plot drift over the acquisition

One line per round (cells, hyb01, ...) at the same x (`fov_id`), so drift at a
FOV shows as vertical spread.

In [ ]:
plot_df = assign_x_positions(cache)

fig, ax = plt.subplots(figsize=(11, 5))

round_ids = sorted(plot_df["round_id"].unique())
cmap      = plt.cm.turbo
color_of  = {rid: cmap(i / max(len(round_ids) - 1, 1)) for i, rid in enumerate(round_ids)}

for rid in round_ids:
    sub = plot_df[plot_df["round_id"] == rid].sort_values("fov_id")
    ax.plot(sub["x"], sub["first_stage_z"], "-o", ms=1, lw=.5, alpha=0.8,
            color=color_of[rid], label=round_label(meta, rid))

ax.set_xlabel("FOV id")
ax.set_ylabel("stage-z (µm) — first frame of each FOV")
ax.set_title("Stage-z drift across the acquisition (every round overlaid at the same FOV x-position)")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
ax.set_ylim([0, 100])  # MF3 nanopositioner's full travel range
fig.subplots_adjust(right=0.78)
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.stage_z_drift.png", dpi=150)
plt.show()

## 5 — Stage-z distribution per round

The same `first_stage_z` values, one violin per round.

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, len(round_ids) * 0.8), 5))

# No data yet: violinplot fails on empty data.
if round_ids:
    violin_data = [plot_df.loc[plot_df["round_id"] == rid, "first_stage_z"].to_numpy() for rid in round_ids]
    parts = ax.violinplot(violin_data, positions=range(len(round_ids)), showmedians=True)
    for body, rid in zip(parts["bodies"], round_ids):
        body.set_facecolor(color_of[rid])
        body.set_alpha(0.8)
    ax.set_xticks(range(len(round_ids)))
    ax.set_xticklabels([round_label(meta, rid) for rid in round_ids], rotation=90)

ax.set_ylabel("stage-z (µm) — first frame of each FOV")
ax.set_title("Stage-z distribution per round")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.stage_z_violin.png", dpi=150)
plt.show()

## 6 — Spatial stage-z heatmap, per round

Each FOV's `first_stage_z` on the stage-position grid, with a diverging
`RdBu` colormap, so a tilt or warp shows as a gradient. `rounds_to_display`
picks the rounds (by `round_label`); two panels per row.

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

rounds_to_display = ['cells', 'hyb01']   # round labels to show

display_round_ids = [
    rid for rid in sorted(cache["round_id"].unique())
    if round_label(meta, rid) in rounds_to_display
]

# One color scale for every panel, so they are comparable.
z_all = cache[cache["round_id"].isin(display_round_ids)]["first_stage_z"]
Z_VMIN, Z_VMAX = z_all.min(), z_all.max()

n_panels = max(len(display_round_ids), 1)
n_cols   = min(2, n_panels)
n_rows   = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5.5 * n_rows), squeeze=False)
axes = axes.flatten()

for ax, round_id in zip(axes, display_round_ids):
    label   = round_label(meta, round_id)
    sub     = cache[cache["round_id"] == round_id]
    fov_ids = sub["fov_id"].tolist()

    grid = positions_to_grid_indices(fov_ids, meta)
    n_x  = max(xi for xi, _ in grid.values()) + 1
    n_y  = max(yi for _, yi in grid.values()) + 1

    matrix = np.full((n_y, n_x), np.nan)
    for fov_id, z in zip(sub["fov_id"], sub["first_stage_z"]):
        xi, yi = grid[fov_id]
        matrix[yi, xi] = z

    im = ax.imshow(matrix, cmap="RdBu", origin="upper", vmin=Z_VMIN, vmax=Z_VMAX)
    ax.set_title(f"Stage-z heatmap — {label}")
    ax.set_xlabel("X grid index  (increasing stage X →)")
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)")
    # Colorbar from the axes divider, so it matches the heatmap's height.
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax, label="stage-z (µm)")

for ax in axes[len(display_round_ids):]:
    ax.axis("off")

fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.stage_z_heatmap.png", dpi=150)
plt.show()